# PractitionerRole — Part 16: Building One Resource in Five Stacks

This is a cookbook notebook, not part of the FHIR-basics tutorial sequence (01-15):
it shows how to construct **one specific `PractitionerRole` resource** in each of the
five languages/engines this project's integrations actually run on:

1. **Python** — this repo's own notebooks/scripts (`Testing.ipynb`, `IntegrationTest.py`).
2. **Java, using HAPI FHIR** — the library most custom FHIR-side Java integration code
   in the NHS genomics space is built on.
3. **JavaScript** — aimed at Mirth Connect (NextGen Connect) Transformer steps, which
   run on Mirth's embedded Rhino JavaScript engine.
4. **TypeScript, using `@types/fhir`** — the DefinitelyTyped FHIR type definitions, for
   a compile-time-checked Node/TypeScript service.
5. **InterSystems ObjectScript** — aimed at InterSystems Health Connect (the
   interoperability engine built on IRIS/Ensemble), the same family of product as
   `FHIR_SERVER` in this repo's `.env`) — shown two ways: plain `%DynamicObject` JSON,
   and IRIS for Health's typed `HS.FHIR.DTL.vR4.Model` FHIR object model.

Unlike most notebooks in this repo, **this one needs no `.env`, no lab network, and no
running server** — only the Python section actually executes here (this repo only has a
Python kernel installed); the Java/JavaScript/TypeScript/ObjectScript sections are complete,
runnable snippets meant to be pasted into *their own* toolchains (a Maven project, a
Mirth Transformer step, an IRIS class), not run inline.

## The target resource

```json
{
  "resourceType": "PractitionerRole",
  "id": "da227a68-8b08-4dc5-ab37-53adbc60811c",
  "practitioner": {
    "identifier": {
      "use": "official",
      "system": "https://fhir.hl7.org.uk/Id/gmc-number",
      "value": "C4706551"
    },
    "display": "Dr Example Clinician"
  },
  "organization": {
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/ods-organization-code",
      "value": "REP"
    }
  }
}
```

The thing worth noticing before writing any code: `practitioner` and `organization` are
both `Reference` types, but neither carries a literal `reference` (a URL/fullUrl to
another resource in the same Bundle) — each carries only an `identifier` (a GMC number,
an ODS code). This is the same **"reference by identifier"** idiom notebook 09 covers for
NHS Digital's own genomic-order examples: don't ship a copy of another system's data,
ship the key it's already known by (GMC/GMP for a person, ODS for an organisation) and
let the receiving system resolve it against the national Spine/PDS/ODS services if it
needs more.

It's worth contrasting this with `Input/FHIR/O21/LiverpoolWomens-1.json`'s own
`PractitionerRole`, which expresses the *same kind* of linkage a different way — a
`contained` `Practitioner`/`Organization` plus a local `#fragment` reference:

```json
{
  "resourceType": "PractitionerRole",
  "contained": [
    {"resourceType": "Practitioner", "id": "0bae4aaf-...",
     "identifier": [{"system": "https://fhir.hl7.org.uk/Id/gmc-number", "value": "C4706551"}]},
    {"resourceType": "Organization", "id": "389f251f-...",
     "identifier": [{"system": "https://fhir.nhs.uk/Id/ods-organization-code", "value": "REP"}]}
  ],
  "organization": {"reference": "#389f251f-..."},
  "practitioner": {"reference": "#0bae4aaf-..."}
}
```

Both are valid FHIR and both round-trip through this repo's `transformToV2`/`transformToFHIR`
endpoints — but the identifier-only form below is shorter, and is the form this notebook
builds in all five languages.

## 1. Python

This repo's own notebooks (see notebook 09) build FHIR resources as plain `dict`s and
serialise with the standard library `json` module — no FHIR modelling library, no
validation beyond what the receiving server/validator does. That's the pattern below.

(`fhirclient` is in `requirements.txt` and used elsewhere in this repo, but only for its
`Binary` model when extracting embedded PDFs — not for general resource construction.)

In [1]:
import json

practitioner_role = {
    "resourceType": "PractitionerRole",
    "id": "da227a68-8b08-4dc5-ab37-53adbc60811c",
    "practitioner": {
        "identifier": {
            "use": "official",
            "system": "https://fhir.hl7.org.uk/Id/gmc-number",
            "value": "C4706551",
        },
        "display": "Dr Example Clinician",
    },
    "organization": {
        "identifier": {
            "system": "https://fhir.nhs.uk/Id/ods-organization-code",
            "value": "REP",
        }
    },
}

print(json.dumps(practitioner_role, indent=2))

{
  "resourceType": "PractitionerRole",
  "id": "da227a68-8b08-4dc5-ab37-53adbc60811c",
  "practitioner": {
    "identifier": {
      "use": "official",
      "system": "https://fhir.hl7.org.uk/Id/gmc-number",
      "value": "C4706551"
    },
    "display": "Dr Example Clinician"
  },
  "organization": {
    "identifier": {
      "system": "https://fhir.nhs.uk/Id/ods-organization-code",
      "value": "REP"
    }
  }
}


A quick self-check: does what we built match the target byte-for-byte in structure
(ignoring key order, which JSON doesn't consider significant)?

In [2]:
target_json = """
{
     "resourceType": "PractitionerRole",
     "id": "da227a68-8b08-4dc5-ab37-53adbc60811c",
     "practitioner": {
      "identifier":  {
       "use": "official",
       "system": "https://fhir.hl7.org.uk/Id/gmc-number",
       "value": "C4706551"
     },
      "display": "Dr Example Clinician"
     },
     "organization": {
       "identifier":  {
       "system": "https://fhir.nhs.uk/Id/ods-organization-code",
       "value": "REP"
     }
     }
   }
"""

assert practitioner_role == json.loads(target_json)
print("matches target")

matches target


To actually send this somewhere, this repo's convention (see `IntegrationTest.py`,
`Testing.ipynb`) is `requests.post(...)` with `verify=False` against a lab server read
from `.env` — a bare `PractitionerRole` isn't normally posted on its own, though; it
shows up either `contained`/referenced inside an order `Bundle` (as in
`LiverpoolWomens-1.json`) or as a directly-addressable resource on a FHIR repository:

```python
import os
import requests
from dotenv import load_dotenv

load_dotenv()
fhir_server = os.getenv("FHIR_SERVER")

r = requests.post(
    f"{fhir_server}PractitionerRole",
    data=json.dumps(practitioner_role),
    headers={"Content-Type": "application/fhir+json"},
    verify=False,
)
```

## 2. Java, using HAPI FHIR

[HAPI FHIR](https://hapifhir.io) is the de facto standard Java FHIR library — it ships a
typed model class per resource/data-type per FHIR version, plus a parser that handles
JSON/XML serialisation.

**Maven dependency** (`pom.xml`) — pin to whatever the current release is on
[hapifhir.io](https://hapifhir.io/hapi-fhir/download.html):

```xml
<dependency>
    <groupId>ca.uhn.hapi.fhir</groupId>
    <artifactId>hapi-fhir-structures-r4</artifactId>
    <version>7.4.0</version>
</dependency>
```

**Building the resource** — the key thing that maps directly onto the "reference by
identifier" idiom above: `org.hl7.fhir.r4.model.Reference` has its own `identifier` field
(`getIdentifier()`/`setIdentifier(Identifier)`) alongside the more commonly-used literal
`reference` string — that's what `.setPractitioner(...)`/`.setOrganization(...)` are
given here, not a `Reference` built from an id string:

```java
import ca.uhn.fhir.context.FhirContext;
import org.hl7.fhir.r4.model.Identifier;
import org.hl7.fhir.r4.model.PractitionerRole;
import org.hl7.fhir.r4.model.Reference;

public class PractitionerRoleExample {

    public static void main(String[] args) {
        PractitionerRole role = new PractitionerRole();
        role.setId("da227a68-8b08-4dc5-ab37-53adbc60811c");

        Identifier practitionerIdentifier = new Identifier()
                .setUse(Identifier.IdentifierUse.OFFICIAL)
                .setSystem("https://fhir.hl7.org.uk/Id/gmc-number")
                .setValue("C4706551");
        role.setPractitioner(new Reference()
                .setIdentifier(practitionerIdentifier)
                .setDisplay("Dr Example Clinician"));

        Identifier organizationIdentifier = new Identifier()
                .setSystem("https://fhir.nhs.uk/Id/ods-organization-code")
                .setValue("REP");
        role.setOrganization(new Reference().setIdentifier(organizationIdentifier));

        FhirContext ctx = FhirContext.forR4();
        String json = ctx.newJsonParser()
                .setPrettyPrint(true)
                .encodeResourceToString(role);

        System.out.println(json);
    }
}
```

`FhirContext.forR4()` is deliberately created once and reused in real applications (it's
expensive to build) — a single `static final FhirContext` on a shared class, not one per
resource, is the usual pattern once this grows beyond a one-off script.

To POST it (mirroring the Python `requests.post` above), HAPI's own
`IGenericClient` (`ctx.newRestfulGenericClient(fhirServerBaseUrl)`, then
`client.create().resource(role).execute()`) is the idiomatic route rather than a bare
`HttpClient` call.

## 3. JavaScript, for Mirth Connect (NextGen Connect)

Mirth Transformer steps run on an embedded Rhino JavaScript engine — plain JS objects
and `JSON.stringify`/`JSON.parse` work exactly as in Node/browser JS, but two things are
Mirth-specific:

- Stick to **ES5 syntax** (`var`, no arrow functions/template literals/`let`/`const`) so
  the same channel code works across older Mirth/Rhino versions, not just the newest.
- Rhino has **LiveConnect** — direct access to Java classes from JavaScript
  (`java.util.UUID.randomUUID()` etc.) — which is how a Transformer step typically
  generates an id without a JS-native UUID library.

A reusable function, dropped into a channel's **Code Templates** so any Transformer step
in the channel can call it:

```javascript
function createPractitionerRole(id, gmcNumber, practitionerDisplay, odsCode) {
    var practitionerRole = {
        resourceType: 'PractitionerRole',
        id: id,
        practitioner: {
            identifier: {
                use: 'official',
                system: 'https://fhir.hl7.org.uk/Id/gmc-number',
                value: gmcNumber
            },
            display: practitionerDisplay
        },
        organization: {
            identifier: {
                system: 'https://fhir.nhs.uk/Id/ods-organization-code',
                value: odsCode
            }
        }
    };
    return practitionerRole;
}
```

Used from a Transformer's **JavaScript Step**, typically to fold into a larger FHIR
Bundle already being assembled in `channelMap`/`msg` rather than sent on its own:

```javascript
var role = createPractitionerRole(
    java.util.UUID.randomUUID().toString(),
    'C4706551',
    'Dr Example Clinician',
    'REP'
);

channelMap.put('practitionerRoleJson', JSON.stringify(role));
```

If the channel is sending straight to a FHIR endpoint, the usual pattern is a **JavaScript
Sender/Destination** whose "HTTP Body" is set to `${practitionerRoleJson}` (or built
directly in a Destination Transformer) with a Content-Type header of
`application/fhir+json` on the HTTP Sender destination's request headers.

## 4. TypeScript, using `@types/fhir`

[`@types/fhir`](https://www.npmjs.com/package/@types/fhir) ships DefinitelyTyped type
definitions for the FHIR resource shapes (R2/R3/R4/R4B/R5, selected via subpath import) —
not a runtime library, just compile-time structural types checked against plain object
literals. The object you build is still a plain JS object at runtime, exactly like the
Mirth example above — `@types/fhir` just gets `tsc` to check it against the
`PractitionerRole` interface shape before it ships, which is the main thing it buys over
the untyped JavaScript/Python versions.

```
npm install --save-dev @types/fhir
```

```typescript
import type { PractitionerRole } from 'fhir/r4';

const practitionerRole: PractitionerRole = {
    resourceType: 'PractitionerRole',
    id: 'da227a68-8b08-4dc5-ab37-53adbc60811c',
    practitioner: {
        identifier: {
            use: 'official',
            system: 'https://fhir.hl7.org.uk/Id/gmc-number',
            value: 'C4706551',
        },
        display: 'Dr Example Clinician',
    },
    organization: {
        identifier: {
            system: 'https://fhir.nhs.uk/Id/ods-organization-code',
            value: 'REP',
        },
    },
};

console.log(JSON.stringify(practitionerRole, null, 2));
```

Because `Reference.identifier` is a real, typed field on the `PractitionerRole` interface
— not a hand-rolled shape — the compiler is what enforces the "reference by identifier"
idiom here: typo `identifer`, or pass a bare string where an `Identifier` object belongs,
and `tsc` rejects it before the code ever runs.

Sending it (e.g. with the global `fetch` in a recent Node runtime) mirrors the Python
example above:

```typescript
const fhirServer = process.env.FHIR_SERVER;

await fetch(`${fhirServer}PractitionerRole`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/fhir+json' },
    body: JSON.stringify(practitionerRole),
});
```

## 5. InterSystems ObjectScript, for Health Connect

Health Connect (like the `FHIR_SERVER` this repo talks to) is built on IRIS. There are two
quite different ways to build this resource in ObjectScript, depending on what's actually
installed on the target: plain IRIS JSON support (available everywhere), or IRIS for
Health's typed FHIR object model (available wherever the FHIR Data Transformation feature
is installed).

### Option A — raw JSON via `%DynamicObject`

IRIS has had native JSON support since Caché 2016.2 via `%DynamicObject`/`%DynamicArray` —
no separate JSON library, no FHIR SDK dependency needed just to build and serialise a JSON
document. This works on any Health Connect/Ensemble/IRIS install, licensed FHIR feature or
not — the resource is just JSON, and IRIS has always been able to build JSON.

```objectscript
ClassMethod CreatePractitionerRole() As %DynamicObject
{
    Set role = {}
    Set role.resourceType = "PractitionerRole"
    Set role.id = "da227a68-8b08-4dc5-ab37-53adbc60811c"

    Set practitionerIdentifier = {}
    Set practitionerIdentifier.use = "official"
    Set practitionerIdentifier.system = "https://fhir.hl7.org.uk/Id/gmc-number"
    Set practitionerIdentifier.value = "C4706551"

    Set practitioner = {}
    Set practitioner.identifier = practitionerIdentifier
    Set practitioner.display = "Dr Example Clinician"
    Set role.practitioner = practitioner

    Set organizationIdentifier = {}
    Set organizationIdentifier.system = "https://fhir.nhs.uk/Id/ods-organization-code"
    Set organizationIdentifier.value = "REP"

    Set organization = {}
    Set organization.identifier = organizationIdentifier
    Set role.organization = organization

    Quit role
}
```

Serialising to a string (e.g. to hand to an `%Net.HttpRequest` as an outbound HTTP
Operation's body) uses `%DynamicAbstractObject`'s own `%ToJSON()`, writing into a
variable passed by reference rather than to the current device:

```objectscript
Set role = ..CreatePractitionerRole()
Set sc = role.%ToJSON(.jsonString)
If $$$ISERR(sc) Quit sc

Set httpRequest = ##class(%Net.HttpRequest).%New()
Set httpRequest.ContentType = "application/fhir+json"
Do httpRequest.EntityBody.Write(jsonString)
Set sc = httpRequest.Post("/PractitionerRole")
```

In a real Health Connect production this would live inside a **Business Operation**
class (e.g. extending `Ens.BusinessOperation`, backed by an `EnsLib.HTTP.OutboundAdapter`)
rather than a bare utility class — `CreatePractitionerRole()` above is the reusable
resource-building piece an operation method would call before handing the JSON to the
adapter.

### Option B — the typed FHIR object model (`HS.FHIR.DTL.vR4.Model`)

If the target is IRIS for Health / HealthShare Health Connect with the shipped FHIR Data
Transformation (DTL) feature installed, there's a second, strongly-typed route:
`HS.FHIR.DTL.vR4.Model.Resource.PractitionerRole` and its related datatype classes,
generated from the FHIR R4 spec itself (class library `HSLIB`) — the same object model
the DTL editor builds under the hood when a transformation maps HL7v2 fields onto FHIR
properties visually. `practitioner`/`organization` are plain properties of type
`HS.FHIR.DTL.vR4.Model.Base.Reference`, and `Reference.identifier` is in turn a plain
property of type `HS.FHIR.DTL.vR4.Model.Datatype.Identifier` — unlike the raw-JSON
version above, a misspelled property name or a value assigned to the wrong data type is
now a compile-time error, not something that only shows up once the JSON reaches a
validator:

```objectscript
ClassMethod CreatePractitionerRoleTyped() As HS.FHIR.DTL.vR4.Model.Resource.PractitionerRole
{
    Set role = ##class(HS.FHIR.DTL.vR4.Model.Resource.PractitionerRole).%New()
    Set role.id = "da227a68-8b08-4dc5-ab37-53adbc60811c"

    Set practitionerIdentifier = ##class(HS.FHIR.DTL.vR4.Model.Datatype.Identifier).%New()
    Set practitionerIdentifier.use = "official"
    Set practitionerIdentifier.system = "https://fhir.hl7.org.uk/Id/gmc-number"
    Set practitionerIdentifier.value = "C4706551"

    Set practitionerRef = ##class(HS.FHIR.DTL.vR4.Model.Base.Reference).%New()
    Set practitionerRef.identifier = practitionerIdentifier
    Set practitionerRef.display = "Dr Example Clinician"
    Set role.practitioner = practitionerRef

    Set organizationIdentifier = ##class(HS.FHIR.DTL.vR4.Model.Datatype.Identifier).%New()
    Set organizationIdentifier.system = "https://fhir.nhs.uk/Id/ods-organization-code"
    Set organizationIdentifier.value = "REP"

    Set organizationRef = ##class(HS.FHIR.DTL.vR4.Model.Base.Reference).%New()
    Set organizationRef.identifier = organizationIdentifier
    Set role.organization = organizationRef

    Quit role
}
```

Notice there's no `Set role.resourceType = "PractitionerRole"` line — `resourceType` is a
read-only computed property (a generated `resourceTypeGet()` method) fixed by the class
itself, one thing this route gets right for free that the `%DynamicObject` version above
has to state explicitly.

Serialising to actual FHIR JSON uses `ToJSON()` — inherited from
`HS.FHIR.DTL.Util.JSON.Adapter`, mixed into every generated resource class — which
returns a `%Stream.Object` rather than a plain string:

```objectscript
Set role = ..CreatePractitionerRoleTyped()
Set jsonStream = role.ToJSON()

Set httpRequest = ##class(%Net.HttpRequest).%New()
Set httpRequest.ContentType = "application/fhir+json"
Do httpRequest.EntityBody.CopyFrom(jsonStream)
Set sc = httpRequest.Post("/PractitionerRole")
```

The reverse direction — parsing FHIR JSON back into a typed object, useful when a DTL
transform needs to read a `PractitionerRole` it received rather than build one — is the
matching class method on `HS.FHIR.DTL.Util.JSON.Adapter`:

```objectscript
Set role = ##class(HS.FHIR.DTL.vR4.Model.Resource.PractitionerRole).FromJSON(jsonStream, "vR4")
```

Reach for Option B when the target already has IRIS for Health's FHIR-DTL feature
available and the resource is complex enough that catching a wrong property name at
compile time earns its keep; reach for Option A when it doesn't, or when the target is a
plain Health Connect/Ensemble install without that feature.

## Summary

Same idiom, five syntaxes — an identifier-only `Reference` is always "an object with one
field, `identifier`, itself an `Identifier`", never a resolvable URL:

| | Resource model | Identifier-only Reference |
|---|---|---|
| **Python** | plain `dict` | `{"identifier": {"system": ..., "value": ...}}` |
| **Java (HAPI FHIR)** | typed `PractitionerRole`/`Reference`/`Identifier` classes | `new Reference().setIdentifier(identifier)` |
| **JavaScript (Mirth/Rhino)** | plain JS object | `{identifier: {system: ..., value: ...}}` |
| **TypeScript (`@types/fhir`)** | `PractitionerRole` interface, plain object at runtime | `{identifier: {system: ..., value: ...}}` (compiler-checked) |
| **ObjectScript (Health Connect, Option A)** | `%DynamicObject` | `Set ref.identifier = identifierObj` |
| **ObjectScript (IRIS for Health, Option B)** | typed `HS.FHIR.DTL.vR4.Model.*` classes | `Set ref.identifier = identifierObj` (compiler-checked) |

Whichever language builds it, the wire format is identical — the target JSON at the top
of this notebook — which is exactly why this pattern is usable as a lingua franca between
a Mirth channel, a Health Connect production, a Java service, a TypeScript service, and
this repo's Python tooling without any of them needing to agree on anything beyond the FHIR spec itself.